In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession
spark_job = SparkSession.builder.appName("Second Spark Project").getOrCreate()

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [ ]:
raw_df = (
    spark_job.read.option("header", "true")
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .csv("salaries.csv")
)
raw_df.show(10, truncate=False)

+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+----------+-------------+--------------------+-----------------+
|   entidadfederativa|      sujetoobligado|              nombre|        denominacion|montoneto|               cargo|                area|montobruto|idInformacion|periodoreportainicio|periodoreportafin|
+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+----------+-------------+--------------------+-----------------+
|             Hidalgo|            Jaltocán|Adolfo Hernandez ...|           Fontanero|   4000.0|           Fontanero|      OBRAS PUBLICAS|    4254.0|     16311845|          01/01/2018|       30/06/2018|
|    Ciudad de México| Secretaría de Salud|ARELY SAMANTA CLE...|"AUXILIAR DE ENFE...| 12177.86|"AUXILIAR DE ENFE...|H.G. ENRIQUE CABRERA|   16092.0|     16480190|          01/01/2018|       31

In [4]:
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: string (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: string (nullable = true)
 |-- periodoreportafin: string (nullable = true)



In [5]:
spark_df.count()

2010281

### DATA CLEANING 

In [6]:
### Checking for blanks
spark_df.distinct().count()

1978329

CHECKING FOR DUPLICATES

In [8]:
df_duplicates = spark_df.groupBy(spark_df.columns).count().filter("count > 1")
df_duplicates.count()

65

In [13]:
df_duplicates.show()

+--------------------+--------------------+--------------------+------------+---------+-----------+--------------------+----------+-------------+--------------------+-----------------+-----+
|   entidadfederativa|      sujetoobligado|              nombre|denominacion|montoneto|      cargo|                area|montobruto|idInformacion|periodoreportainicio|periodoreportafin|count|
+--------------------+--------------------+--------------------+------------+---------+-----------+--------------------+----------+-------------+--------------------+-----------------+-----+
|           Chihuahua|Junta Municipal d...|                NULL|        NULL|     NULL|       NULL|                NULL|      NULL|         NULL|                NULL|             NULL|  275|
|     Baja California|Sistema para el D...|Rosa Angélica Tap...| Intendencia|   5897.6|Intendencia|Departamento de C...|      NULL|         NULL|                NULL|             NULL|    2|
|             Sinaloa|AY00200-Ayuntamie...|  

In [9]:
df_non_duplicates = spark_df.dropDuplicates()
df_non_duplicates.count()

1978329

In [11]:
duplicates_count = spark_df.count() - df_duplicates.count()
duplicates_count

2010216

Checking For NaN rows

In [16]:
spark_df.na.drop().count()

1822306

### Setting Not Null to at least 8 values

In [27]:
spark_df.na.drop(how="any", thresh=8).count()

# subset_cols = spark_df.columns[0:7]
# spark_df.na.drop(thresh=5, subset=subset_cols).count()

1923815

In [28]:
spark_df = spark_df.na.drop(how="any", thresh=8)
spark_df.count()

1923815

### Data Validatuon

changing the montoneto and montobruto columns from string dtype to integer

and converting both periodoreportaincio and periodoreportafin from string to datetime

In [ ]:
from pyspark.sql import functions as F

# Validate salary strings before casting and parse both supported date formats.
clean_df = (
    raw_df
    .withColumn(
        "montoneto",
        F.when(
            F.trim(F.col("montoneto")).rlike(r"^[0-9,.\s-]+$"),
            F.regexp_replace(F.trim(F.col("montoneto")), ",", "").cast("double")
        )
    )
    .withColumn(
        "montobruto",
        F.when(
            F.trim(F.col("montobruto")).rlike(r"^[0-9,.\s-]+$"),
            F.regexp_replace(F.trim(F.col("montobruto")), ",", "").cast("double")
        )
    )
    .withColumn(
        "periodoreportainicio",
        F.coalesce(
            F.to_date(F.trim(F.col("periodoreportainicio")), "dd/MM/yyyy"),
            F.to_date(F.trim(F.col("periodoreportainicio")), "yyyy-MM-dd")
        )
    )
    .withColumn(
        "periodoreportafin",
        F.coalesce(
            F.to_date(F.trim(F.col("periodoreportafin")), "dd/MM/yyyy"),
            F.to_date(F.trim(F.col("periodoreportafin")), "yyyy-MM-dd")
        )
    )
    .filter(
        F.col("montoneto").isNotNull()
        & F.col("montobruto").isNotNull()
        & F.col("periodoreportainicio").isNotNull()
        & F.col("periodoreportafin").isNotNull()
    )
    .withColumn(
        "period_dias",
        F.datediff("periodoreportafin", "periodoreportainicio")
    )
)

spark_df = clean_df
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- periodo_dias: integer (nullable = true)



### FEATURE ENGINEERING
Creating a new column to calculate report interval in days which is period_dias in mexico

In [42]:
spark_df = spark_df.drop('periodo_dias')

In [43]:
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)



In [ ]:
spark_df = spark_df.withColumn('period_dias', (spark_df['periodoreportafin'] - spark_df['periodoreportainicio']))

In [48]:
from pyspark.sql.functions import to_date

spark_df = (
    spark_df
    .withColumn(
        "periodoreportainicio",
        to_date("periodoreportainicio", "dd/MM/yyyy")
    )
    .withColumn(
        "periodoreportafin",
        to_date("periodoreportafin", "dd/MM/yyyy")
    )
)

In [50]:
from pyspark.sql.functions import datediff

spark_df = spark_df.withColumn(
    "period_dias",
    datediff("periodoreportafin", "periodoreportainicio")
)

In [51]:
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)



In [54]:
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)



In [55]:
spark_df.select(
    "periodoreportainicio",
    "periodoreportafin",
    "period_dias"
).show(10, truncate=False)

{"ts": "2026-09-05 09:34:54.666", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '01/01/2018' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "104)", "fragment": "to_date", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o530.showString.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '01/01/2018' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"to_dat

DateTimeException: [CAST_INVALID_INPUT] The value '01/01/2018' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"to_date" was called from
java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java:104)


### EDA WITH SPARK 

### DATA INFO IN ENGLISH
| Feature                | Data type | Meaning                                                                                                                                                                                                                           |
| ---------------------- | --------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `entidadfederativa`    | string    | **Federal entity/state**. This identifies the Mexican state or federal entity where the public institution/person belongs. Examples could include Jalisco, Oaxaca, Ciudad de México, etc.                                         |
| `sujetoobligado`       | string    | **Obligated entity**. This refers to the government institution or public body that is legally required to disclose the information. Think of it as the organization/employer responsible for the record.                         |
| `nombre`               | string    | **Name**. The name of the person associated with the record, presumably the public employee/official.                                                                                                                             |
| `denominacion`         | string    | **Denomination/name of the position or information category**. This generally describes the formal designation associated with the record. You should inspect actual values to determine exactly how this dataset uses the field. |
| `montoneto`            | double    | **Net amount**. The amount received after applicable deductions. This is a numerical monetary feature, so you can calculate averages, totals, minimums, maximums, etc.                                                            |
| `cargo`                | string    | **Position/job title**. Describes the person's official role or position. Examples might be Director, Coordinator, Analyst, Secretary, etc.                                                                                       |
| `area`                 | string    | **Department/area**. The organizational department or administrative area where the person works.                                                                                                                                 |
| `montobruto`           | double    | **Gross amount**. The amount before deductions. This is another numerical monetary feature and will be particularly useful for comparing gross vs. net compensation.                                                              |
| `idInformacion`        | string    | **Information ID/record identifier**. A unique or semi-unique identifier associated with the information record. This is useful for checking duplicates and data integrity.                                                       |
| `periodoreportainicio` | date      | **Reporting period start date**. The date on which the reported period begins.                                                                                                                                                    |
| `periodoreportafin`    | date      | **Reporting period end date**. The date on which the reported period ends.                                                                                                                                                        |
